# Nishimoto and Gallant (2011) on CRCNS mt-2: what the spatial pooling costs

Paper: Nishimoto S, Gallant JL (2011), J Neurosci 31(41):14551-14564,
doi:10.1523/jneurosci.6801-10.2011. Dataset: CRCNS mt-2 (doi:10.6080/K0DN4374), read from the
public bucket `crcnsarchive/mt-2/live/` (unversioned; `reproduce_nishimoto_2011.ipynb` prints the
MD5 sums of the files). Claims: `claims/nishimoto-2011.yaml`.

The main notebook fits every model with the regression weights pooled over space onto 763
regressors (3 and 6 cycles per cRF onto 2 x 2 regions), because the paper's model, with a weight
for each of its 5956 filters, does not fit in the time limit for 12 models and 45 neurons. This
notebook measures what that costs for the compressive and divisive model, the one the claims
below rest on. It refits that model for every neuron with 3 and 6 cycles per cRF pooled onto
3 x 3 regions and 1.5 cycles per cRF at each of its four positions (1465 regressors), and for
three neurons with 4 x 4 regions and every simple channel of the bank (3085 regressors), closer
to the paper's model. The delay offset of each neuron is chosen as in the main notebook, by the
held-out error of the 2 x 2 model on the training data. Everything else (filter bank, features,
boosting, validation) is as in the main notebook (`mt2.py`, `pooling_neuron`).

| Claim | Tested here |
|---|---|
| fig3b-mean-prediction | with the finer pooling |
| fig3a-example-neuron | with the finer pooling |
| complex-cells-dominate | with the finer pooling |
| natural-movie-control | with the finer pooling |

In [ ]:
import os, time, resource
os.environ.setdefault("OMP_NUM_THREADS", "2")
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from scipy.stats import wilcoxon
import mt2

T0 = time.time()
sizes = mt2.list_sizes()
names = sorted(sizes)
FINE = ["ct0002_arg0466d_128.mat", "ct0005_arg0466d_128.mat", "ct0006_arg0466d_128.mat"]
N_RUN = int(os.environ.get("MT2_N", len(names)))  # a subset while developing
order = sorted(names, key=lambda n: -sizes[n]) if N_RUN >= len(names) else sorted(names, key=lambda n: sizes[n])[:N_RUN]
t = time.time()
with ProcessPoolExecutor(2) as ex:
    futs = {n: ex.submit(mt2.pooling_neuron, n, n in FINE, 2) for n in order}
    R = {n: f.result() for n, f in futs.items()}
names_run = [n for n in names if n in R]
print(f"{len(R)} neurons in {(time.time() - t) / 60:.1f} min; per neuron {np.median([R[n]['t_total'] for n in R]):.0f} s median")
print("peak memory of a worker:", resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1e6, "GB")

## Data check

In [ ]:
def acc(r):
    return float(np.mean(list(r.values())))

IDX = [n[:6] for n in names_run]
monkey = np.array([R[n]["cellid"][0] for n in names_run])
main = pd.Series([acc(R[n]["r2x2"]) for n in names_run], index=IDX)
p3 = pd.Series([acc(R[n]["p3x3"]["r"]) for n in names_run], index=IDX)
print("neurons:", len(names_run), "; by monkey:", dict(zip(*np.unique(monkey, return_counts=True))))
print("validation sets:", dict(zip(*np.unique(["+".join(R[n]["sets"]) for n in names_run], return_counts=True))))
print("delay offsets (frames earlier):", dict(zip(*np.unique([R[n]["offset"] for n in names_run], return_counts=True))))
print(f"2 x 2 pooling at the chosen offsets: mean r {main.mean():.3f}")
print("regressors with 3 x 3 pooling:", R[names_run[0]]["p3x3"]["n_complex"], "complex,", R[names_run[0]]["p3x3"]["n_simple"], "simple")

## fig3b-mean-prediction

> The compressive and divisive V1 filter model predicts the validation responses of the 52 MT
> neurons with an average correlation of r = 0.52. (Expected 0.52 +- 0.052, over the 45 neurons
> in the dataset.)

A neuron's accuracy is the mean of its two validation correlations where it has both sets, as in
the main notebook.

In [ ]:
p3e = pd.Series([R[n]["p3x3"]["r"]["enhanced"] for n in names_run], index=IDX)
print(f"pooling 3 x 3: mean r {p3.mean():.3f} (median {p3.median():.3f}, range {p3.min():.3f} to {p3.max():.3f}); "
      f"within 0.468 to 0.572: {0.468 <= p3.mean() <= 0.572}; motion-enhanced only {p3e.mean():.3f}")
print(f"pooling 2 x 2: mean r {main.mean():.3f}")
print(f"3 x 3 better than 2 x 2 in {(p3 > main).sum()} of {len(p3)}, mean difference {(p3 - main).mean():+.3f}, "
      f"Wilcoxon p = {wilcoxon(p3, main).pvalue:.3g}; without ct0133 {p3[p3.index != 'ct0133'].mean():.3f}")
for mk in np.unique(monkey):
    print(f"  monkey {mk}: n = {(monkey == mk).sum()}, mean r {p3[monkey == mk].mean():.3f}")
print(f"neurons between 0.2 and 0.8 (Figure 3B span): {((p3 >= 0.2) & (p3 <= 0.8)).sum()} of {len(p3)}; "
      f"below 0.2: {(p3 < 0.2).sum()}, above 0.8: {(p3 > 0.8).sum()}")
its = np.array([R[n]["p3x3"]["its"] for n in names_run])
print(f"boosting steps per fold: median {np.median(its):.0f}, range {its.min()} to {its.max()} (cap 20000)")
fine_n = [n for n in names_run if "fine" in R[n]]
if fine_n:
    nf = R[fine_n[0]]["fine"]["n_complex"] + R[fine_n[0]]["fine"]["n_simple"]
    print(pd.DataFrame({"2x2 (763)": [main[n[:6]] for n in fine_n], "3x3 (1465)": [p3[n[:6]] for n in fine_n],
                        f"4x4 + all simple ({nf})": [acc(R[n]["fine"]["r"]) for n in fine_n]},
                       index=[n[:6] for n in fine_n]).round(3).to_string())

**Result: reproduced with the finer pooling.** With 3 x 3 regions (1465 regressors) the compressive and divisive model predicts the 45 neurons with a mean correlation of 0.518 (median 0.507, range 0.263 to 0.761), within 0.468 to 0.572; with the motion-enhanced set only it is 0.524. Every neuron is better predicted than with the 2 x 2 pooling of the main notebook (45 of 45, mean +0.070, Wilcoxon p = 5.7e-14), which gives 0.448. Monkey f 0.524, monkey v 0.500; all 45 neurons lie between 0.2 and 0.8, as in Figure 3B. For the three neurons fitted with the finest pooling (3085 regressors) the correlation is 0.614, 0.352 and 0.347, against 0.636, 0.398 and 0.322 with 3 x 3 regions, so the finer pooling does not raise it further on these three.

## fig3a-example-neuron

> For one example neuron the correlation between model predictions and observed responses is
> r = 0.72 (expected 0.72 +- 0.072; the paper's neuron had 10 validation repeats).

In [ ]:
hit = p3[(p3 >= 0.648) & (p3 <= 0.792)]
print(f"pooling 3 x 3: neurons with r in 0.648 to 0.792: {len(hit)}", hit.round(3).to_dict())
print("the two neurons whose validation set shows 10 repeats (main notebook):", {k: round(p3[k], 3) for k in ("ct0028", "ct0030") if k in p3})

**Result: reproduced.** With the finer pooling four neurons fall within 0.648 to 0.792 (ct0013 0.758, ct0021 0.761, ct0024 0.699, ct0039 0.760). The two neurons whose validation set shows 10 repeats give 0.403 (ct0028) and 0.492 (ct0030), so the paper's example neuron is still not identified.

## complex-cells-dominate

> Most of the response variance is captured by the model complex cells, and the model simple
> cells have only a small effect on prediction performance. (Reproduced when the complex-only
> model reaches at least half of the full model's mean squared correlation and drops the mean
> correlation less than the simple-only model does.)

In [ ]:
co3 = pd.Series([acc(R[n]["p3x3"]["complex_only"]) for n in names_run], index=IDX)
so3 = pd.Series([acc(R[n]["p3x3"]["simple_only"]) for n in names_run], index=IDX)
print(f"pooling 3 x 3: full {p3.mean():.3f}, complex only {co3.mean():.3f} (r^2 ratio {(co3**2).mean() / (p3**2).mean():.3f}), "
      f"simple only {so3.mean():.3f} (r^2 ratio {(so3**2).mean() / (p3**2).mean():.3f})")
print(f"drop of mean r: complex only {p3.mean() - co3.mean():.4f}, simple only {p3.mean() - so3.mean():.4f}")
for n in fine_n:
    f_ = R[n]["fine"]
    print(f"finest pooling, {n[:6]} ({f_['n_simple']} simple regressors, every position, up to 1.5 cycles per cRF): "
          f"full {acc(f_['r']):.3f}, complex only {acc(f_['complex_only']):.3f}, simple only {acc(f_['simple_only']):.3f}")

**Result: reproduced.** With the finer pooling the complex-only model gives a mean correlation of 0.514 against 0.518 for the full model (ratio of mean squared correlations 0.985), and the simple-only model 0.194 (ratio 0.191). With every simple channel of the bank (1082 simple regressors), the simple-only model gives 0.367, 0.078 and 0.141 for the three neurons, against 0.622, 0.364 and 0.333 for complex only and 0.614, 0.352 and 0.347 for the full model.

## natural-movie-control

> Models fit on motion-enhanced movies predict responses to natural movies without motion
> enhancement as well as responses to new motion-enhanced movies (mean r = 0.533 against 0.537,
> 15 neurons; Wilcoxon signed-rank p > 0.10).

In [ ]:
nn = [n for n in names_run if "natural" in R[n]["p3x3"]["r"]]
rn3 = np.array([R[n]["p3x3"]["r"]["natural"] for n in nn])
re3 = np.array([R[n]["p3x3"]["r"]["enhanced"] for n in nn])
if nn:
    print(f"{len(nn)} neurons with both validation sets")
    print(f"pooling 3 x 3: natural {rn3.mean():.3f} [within 0.533 +- 0.053: {abs(rn3.mean() - 0.533) <= 0.053}], "
          f"motion-enhanced {re3.mean():.3f} [within 0.537 +- 0.054: {abs(re3.mean() - 0.537) <= 0.054}], "
          f"Wilcoxon p = {wilcoxon(rn3, re3).pvalue:.4g}; natural higher in {(rn3 > re3).sum()} of {len(nn)}; mean difference {np.mean(rn3 - re3):+.3f}")
    for n, a_, b_ in zip(nn, rn3, re3):
        print(f"  {n[:6]}  natural {a_:.3f}  enhanced {b_:.3f}")

**Result: reproduced with the finer pooling.** For the 14 neurons with a natural-movie validation set, the mean correlation is 0.523 for natural movies and 0.563 for motion-enhanced movies, both within tolerance (0.533 +- 0.053 and 0.537 +- 0.054), and the Wilcoxon signed-rank test finds no difference (p = 0.43; natural higher in 5 of 14).

## Summary

| Claim | With the finer pooling (3 x 3 regions, 1465 regressors) |
|---|---|
| fig3b-mean-prediction | reproduced (mean r 0.518; 0.448 with the 2 x 2 pooling) |
| fig3a-example-neuron | reproduced (4 neurons within 0.648 to 0.792) |
| complex-cells-dominate | reproduced |
| natural-movie-control | reproduced (0.523 natural, 0.563 motion-enhanced, p = 0.43) |

The finer pooling raises the prediction accuracy of every neuron, by 0.070 on average, and brings
the population mean within the paper's tolerance. The other models of Figure 2 and the spectral
analyses of Figures 6 to 8 were fitted only with the 2 x 2 pooling (main notebook), so their
effect sizes may also be underestimated there. This notebook differs from the paper's method in
the same ways as the main notebook, except for the finer pooling.

In [ ]:
print(f"notebook run time {(time.time() - T0) / 60:.1f} min")